# 05. Historical price returns and risk

**Research question:** What price returns and risks did investors experience through 31 December 2025?

These are descriptive histories, not investable signals or forecasts. The cleaned MGS close is used. Daily historical dividend payments are unavailable, so all returns below are **price only**.

## ⭐ 1. Inputs and conventions

Load the approved cleaned CSVs. All windows end on **31 December 2025**: three years start 1 January 2023, five years start 1 January 2021, and full history starts with the first supplied trading day. A window uses its first observed close, which may be the first trading day after its calendar start. No missing price is forward filled.

Daily simple return is `close / prior close − 1`. Arithmetic annualised return is mean daily return × **252 trading days**. CAGR uses first and last close and actual elapsed calendar years. Volatility is sample standard deviation of daily returns × √252. Sharpe uses a **3% annual risk-free assumption**, converted to an effective daily rate. Beta and correlation use only dates paired with the supplied benchmark. Drawdown is the fall from the preceding running peak; maximum drawdown is its most negative value. Benchmark-relative return compares cumulative company and benchmark price returns over the same endpoints.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_cleaned_data
from src.research.price_risk import (daily_simple_returns, drawdown_series,
    price_panel, risk_summary, rolling_annualised_volatility)
annual, prices = load_cleaned_data(project_root / "data" / "Cleaned")
as_of = "2025-12-31"
company_info = annual.loc[annual["year"].eq(2025), ["ticker", "company", "sector"]].copy()
assert len(company_info) == 20 and company_info["ticker"].is_unique
display(company_info.groupby("sector").size().rename("companies").to_frame())

## 2. Three comparison windows

The table keeps all 20 companies in every window. Percentages are displayed in percentage points for readability; calculation functions retain fractional values. A blank metric means that the data cannot support that calculation.

In [ ]:
window_starts = {"3 years": "2023-01-01", "5 years": "2021-01-01", "Full history": None}
window_results = []
for window, start_date in window_starts.items():
    result = risk_summary(prices, start=start_date, end=as_of, risk_free_rate=0.03)
    result.insert(0, "window", window)
    window_results.append(result)
risk_by_window = pd.concat(window_results, ignore_index=True).merge(company_info, on="ticker", validate="many_to_one")
assert risk_by_window.groupby("window")["ticker"].nunique().eq(20).all()
percent_columns = ["total_price_return", "arithmetic_annualised_return", "cagr",
                   "annualised_volatility", "max_drawdown", "benchmark_relative_return"]
display_columns = ["window", "ticker", "company", "sector", "total_price_return",
                   "arithmetic_annualised_return", "cagr", "annualised_volatility",
                   "max_drawdown", "sharpe", "beta", "benchmark_correlation",
                   "benchmark_relative_return"]
display_table = risk_by_window[display_columns].copy()
display_table[percent_columns] = display_table[percent_columns].mul(100)
display(display_table.round(2).rename(columns={
    "total_price_return": "Total price return (%)", "arithmetic_annualised_return": "Arithmetic annual return (%)",
    "cagr": "CAGR (%)", "annualised_volatility": "Volatility (% annual)",
    "max_drawdown": "Maximum drawdown (%)", "sharpe": "Sharpe (3% risk-free)",
    "beta": "Beta vs benchmark", "benchmark_correlation": "Benchmark correlation",
    "benchmark_relative_return": "Relative price return (%)"}))

**Reading the table:** CAGR and arithmetic annualised return answer different questions and need not match. A severe maximum drawdown shows a historical peak-to-trough loss, regardless of the final cumulative return. Sharpe and beta are sample estimates and can change substantially across windows.

## 3. Normalised price paths by sector

Each panel shows four companies in one sector plus the benchmark, all rebased to 100 at the first trading day of the three-year window. Splitting sectors prevents a single crowded 20-line chart. These are prices, not total-return indices.

In [ ]:
panel_3y = price_panel(prices, start="2023-01-01", end=as_of)
sector_groups = company_info.groupby("sector")["ticker"].apply(list)
fig, axes = plt.subplots(len(sector_groups), 1, figsize=(12, 16), sharex=True)
for ax, (sector, tickers) in zip(axes, sector_groups.items()):
    subset = panel_3y[tickers + ["BENCHMARK"]]
    rebased = subset.div(subset.iloc[0]).mul(100)
    for ticker in tickers:
        ax.plot(rebased.index, rebased[ticker], label=ticker, linewidth=1.4)
    ax.plot(rebased.index, rebased["BENCHMARK"], label="Benchmark", color="black", linestyle="--", linewidth=1)
    ax.set_title(sector)
    ax.set_ylabel("Index (start = 100)")
    ax.grid(alpha=.25)
    ax.legend(ncol=5, fontsize=8, loc="upper left")
axes[-1].set_xlabel("Date")
fig.suptitle("Three-year normalised closing prices by sector", y=1.005)
fig.tight_layout()
plt.show()

## ⭐ 4. Return versus risk

This scatter uses the five-year window. Horizontal position is realised annualised volatility, and vertical position is historical CAGR. Colour marks sector; position does not establish a superior investment.

In [ ]:
five_year = risk_by_window.loc[risk_by_window["window"].eq("5 years")].copy()
fig, ax = plt.subplots(figsize=(10, 7))
for sector, group in five_year.groupby("sector"):
    ax.scatter(group["annualised_volatility"] * 100, group["cagr"] * 100, label=sector, s=55, alpha=.8)
for row in five_year.itertuples():
    ax.annotate(row.ticker, (row.annualised_volatility * 100, row.cagr * 100),
                xytext=(4, 3), textcoords="offset points", fontsize=8)
ax.axhline(0, color="grey", linewidth=.8)
ax.set(title="Five-year historical CAGR versus price volatility", xlabel="Annualised volatility (%)", ylabel="Price CAGR (%)")
ax.grid(alpha=.25)
ax.legend(title="Sector", fontsize=8)
plt.show()

## ⭐ 5. Drawdowns and changing volatility

Maximum drawdown compares each company's worst observed decline within the five-year window. Rolling volatility uses a **63-session** sample, approximately one quarter, and is shown for one alphabetically selected company per sector to keep the chart legible. The selected companies are illustrations, not recommendations.

In [ ]:
ordered = five_year.sort_values(["sector", "max_drawdown", "ticker"])
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(ordered["ticker"], ordered["max_drawdown"] * 100, color="#9a4f54")
ax.set(title="Five-year maximum price drawdown by company", xlabel="Maximum drawdown (%)", ylabel="Ticker")
ax.grid(axis="x", alpha=.25)
plt.show()

illustrative = company_info.sort_values("ticker").groupby("sector").head(1)["ticker"].tolist()
rolling_vol = rolling_annualised_volatility(panel_3y[illustrative], sessions=63)
fig, ax = plt.subplots(figsize=(11, 5))
for ticker in illustrative:
    ax.plot(rolling_vol.index, rolling_vol[ticker] * 100, label=ticker)
ax.set(title="Three-year rolling 63-session annualised volatility: one ticker per sector",
       xlabel="Date", ylabel="Annualised volatility (%)")
ax.legend(title="Ticker")
ax.grid(alpha=.25)
plt.show()

## 6. Benchmark-relative performance

The next chart plots the company-to-benchmark ratio, rebased to 100 at the first three-year close. A rising line means the company price outpaced the benchmark price over that interval; it says nothing about dividends or future relative performance.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
relative = panel_3y[illustrative].div(panel_3y["BENCHMARK"], axis=0)
relative = relative.div(relative.iloc[0]).mul(100)
for ticker in illustrative:
    ax.plot(relative.index, relative[ticker], label=ticker)
ax.axhline(100, color="grey", linestyle="--", linewidth=.8)
ax.set(title="Three-year price performance relative to supplied benchmark",
       xlabel="Date", ylabel="Relative index (start = 100)")
ax.legend(title="Ticker")
ax.grid(alpha=.25)
plt.show()

## ⭐ 7. Observations and limitations

The following statements are generated from the five-year table so they remain tied to the displayed calculations. They describe the historical sample only.

In [ ]:
most_volatile = five_year.loc[five_year["annualised_volatility"].idxmax()]
deepest_drawdown = five_year.loc[five_year["max_drawdown"].idxmin()]
print(f"Highest observed five-year volatility: {most_volatile['ticker']} ({most_volatile['annualised_volatility']:.1%} annualised).")
print(f"Deepest five-year drawdown: {deepest_drawdown['ticker']} ({deepest_drawdown['max_drawdown']:.1%}).")
print("The comparison includes 20 companies; all return measures exclude dividends.")

**Limitations and follow-up:** The 2025 close and all other historical prices are viewed with hindsight. These charts do not test an investable signal and therefore do not use annual fundamentals before their `available_date`. The 3% risk-free assumption is fixed across periods. Betas, Sharpe ratios and maximum drawdowns depend on window choice; compare all three windows before drawing conclusions. The supplied benchmark is a price series, and daily dividends are unavailable. Examine the business and valuation evidence in later notebooks before any investment decision.